<a href="https://colab.research.google.com/github/ale66/learn-datascience/blob/main/week-9/Ranking_world_cup/2026_FIFA_World_Cup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2026 FIFA World Cup Simulation and Team Rating Analysis

This notebook simulates the 2026 FIFA World Cup using actual group data. It includes:
*   **Team Rating Systems:** Massey, Keener, and Markov.
*   **Simulation Phases:** Group stage advancement and direct elimination knockout rounds.
*   **Objective:** Predict match outcomes and determine a tournament winner.

## 1: Download + Read Data

The dataset `results.csv` is obtained from the 'International Football Results from 1872 to 2026' Kaggle dataset, which compiles international football match results.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving results.csv to results.csv


In [2]:
import pandas as pd

df = pd.read_csv("results.csv")
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


Take historical data of the last 8 years, which contains two previous world cups

In [3]:
df_8y = df[df["date"] >= "2018-01-01"]

In [4]:
print(df_8y.shape)
df_8y.head()

(8180, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
41297,2018-01-02,Iraq,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41298,2018-01-02,Oman,Bahrain,1.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41299,2018-01-05,Oman,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41300,2018-01-07,Estonia,Sweden,1.0,1.0,Friendly,Abu Dhabi,United Arab Emirates,True
41301,2018-01-11,Denmark,Sweden,0.0,1.0,Friendly,Abu Dhabi,United Arab Emirates,True


## 2026 FIFA World Cup group

This section fetches the 2026 FIFA World Cup group stage data from a public JSON file. It then processes this data to extract the teams participating in each group, which will be used for the simulation.

In [5]:
import requests

url = "https://raw.githubusercontent.com/openfootball/worldcup.json/master/2026/worldcup.json"
data = requests.get(url).json()

extract teams + groups

In [6]:
groups = {}
teams = set()

for match in data["matches"]:

    team1 = match.get("team1")
    team2 = match.get("team2")
    group = match.get("group")

    # skip knockout placeholders like W101 etc.
    if not team1 or not team2:
        continue

    # only group stage matches
    if group is None:
        continue

    if "Group" not in group:
        continue

    if group not in groups:
        groups[group] = set()

    groups[group].add(team1)
    groups[group].add(team2)

    teams.add(team1)
    teams.add(team2)

clean groups

In [7]:
groups = {k: sorted(list(v)) for k, v in groups.items()}
teams = sorted(list(teams))

sanity check

In [8]:
print("number of teams:", len(teams))
print("number of groups:", len(groups))

for g, t in groups.items():
    print(g, t)

number of teams: 48
number of groups: 12
Group A ['Czech Republic', 'Mexico', 'South Africa', 'South Korea']
Group B ['Bosnia & Herzegovina', 'Canada', 'Qatar', 'Switzerland']
Group C ['Brazil', 'Haiti', 'Morocco', 'Scotland']
Group D ['Australia', 'Paraguay', 'Turkey', 'USA']
Group E ['Curaçao', 'Ecuador', 'Germany', 'Ivory Coast']
Group F ['Japan', 'Netherlands', 'Sweden', 'Tunisia']
Group G ['Belgium', 'Egypt', 'Iran', 'New Zealand']
Group H ['Cape Verde', 'Saudi Arabia', 'Spain', 'Uruguay']
Group I ['France', 'Iraq', 'Norway', 'Senegal']
Group J ['Algeria', 'Argentina', 'Austria', 'Jordan']
Group K ['Colombia', 'DR Congo', 'Portugal', 'Uzbekistan']
Group L ['Croatia', 'England', 'Ghana', 'Panama']


Pick up the corresponding dataset

In [9]:
df_48 = df[
    (df["home_team"].isin(teams)) &
    (df["away_team"].isin(teams))
].copy()

## 2: Massey Ratings (Linear Algebra Version)
Core Formula: Mr=p

*   M = match matrix
*   r = ratings
*   p = goal difference

### Construct Massey Matrix

1. Teams

In [10]:
import numpy as np
import pandas as pd

# Identify all unique teams that actually played in df_48
actual_playing_teams_in_df_48 = set(df_48["home_team"]).union(set(df_48["away_team"]))

# Filter the original 'teams' list (from World Cup 2026 data)
teams = sorted(list(t for t in teams if t in actual_playing_teams_in_df_48))

n = len(teams) # Update n to reflect the new number of teams
team_index = {t:i for i,t in enumerate(teams)}

2. Fill Matrix

In [11]:
M = np.zeros((n, n))
p = np.zeros(n)

for _, row in df_48.iterrows():
    home = team_index[row["home_team"]]
    away = team_index[row["away_team"]]

    hs = row["home_score"]
    as_ = row["away_score"]

    if np.isnan(hs) or np.isnan(as_):
        continue

    diff = hs - as_

    # Massey matrix
    M[home, home] += 1
    M[away, away] += 1
    M[home, away] -= 1
    M[away, home] -= 1

    p[home] += diff
    p[away] -= diff

3. Constraint (Very Important)

In [12]:
M[-1, :] = 1
p[-1] = 0

4. Solve

In [13]:
ratings_massey = np.linalg.solve(M, p)

5. Output

In [14]:
massey_df = pd.DataFrame({
    "team": teams,
    "massey": ratings_massey
})

massey_df = massey_df.sort_values("massey", ascending=False)
massey_df.reset_index(drop=True, inplace=True)

massey_df.head(20)

,team,massey
0,Brazil,1.534401
1,England,1.386221
2,Spain,1.339730
3,Germany,1.325446
4,Argentina,1.252077
5,Netherlands,0.979249
6,France,0.840904
7,Sweden,0.780785
8,Uruguay,0.775020
9,Portugal,0.709379


## 3: Keener Ratings (Eigenvector)
Core Idea:
Ar=λr

In [15]:
A = np.zeros((n, n))

# Filter out rows with NaN scores from df_48 for Keener ratings calculation
df_48_keener = df_48.dropna(subset=["home_score", "away_score"]).copy()

for _, row in df_48_keener.iterrows():
    i = team_index[row["home_team"]]
    j = team_index[row["away_team"]]

    A[i, j] += row["home_score"] + 1
    A[j, i] += row["away_score"] + 1

# normalize
col_sum = A.sum(axis=0)

# Avoid division by zero for teams that might not have played any match
# Replace 0 with 1 in col_sum where it's 0 to prevent NaN/Inf in A after division
col_sum[col_sum == 0] = 1
A = A / col_sum

### Calculate Eigenvector

In [16]:
# Ensure A is not all zeros or contains NaNs
# Also handle cases where A might contain inf
if not np.all(A == 0) and not np.any(np.isnan(A)) and not np.any(np.isinf(A)):
    eigvals, eigvecs = np.linalg.eig(A)

    idx = np.argmax(eigvals.real)

    keener_ratings = eigvecs[:, idx].real

    # Key: Standardize (non-negative + normalize)
    keener_ratings = np.abs(keener_ratings)
    keener_ratings = keener_ratings / keener_ratings.sum()

    keener_df = pd.DataFrame({
        "team": teams,
        "rating": keener_ratings
    }).sort_values("rating", ascending=False)

    print(keener_df.head(10))
else:
    print("Matrix A is invalid (all zeros, contains NaN, or Inf values). Cannot compute Keener ratings.")

           team    rating
1     Argentina  0.060435
5        Brazil  0.060294
17      Germany  0.051066
15      England  0.048949
44      Uruguay  0.048742
27  Netherlands  0.046125
4       Belgium  0.043855
16       France  0.043097
40       Sweden  0.038836
41  Switzerland  0.035658


## 4. Markov Ratings

### Construct Markov Transition Matrix

Markov ratings model the strength of teams based on a Markov chain, where teams are states and the transitions between states are determined by match outcomes (goal scores). The core idea is to establish a transition matrix where $P_{ij}$ represents the 'strength' or 'likelihood' of team $i$ transitioning to or dominating team $j$. The stationary distribution of this Markov chain then provides the long-term ratings of each team.

In [17]:
A_markov = np.zeros((n, n))

# Filter out rows with NaN scores from df_48 for Markov ratings calculation
df_48_markov = df_48.dropna(subset=["home_score", "away_score"]).copy()

for _, row in df_48_markov.iterrows():
    i = team_index[row["home_team"]]
    j = team_index[row["away_team"]]

    # A[i, j] means team i's strength against team j
    A_markov[i, j] += row["home_score"] + 1 # home team scores against away team
    A_markov[j, i] += row["away_score"] + 1 # away team scores against home team

# Normalize the matrix to be row-stochastic (each row sums to 1)
row_sums = A_markov.sum(axis=1)

# Avoid division by zero for teams that might not have played any match or scored any goals
row_sums[row_sums == 0] = 1

P_markov = A_markov / row_sums[:, np.newaxis]

### Calculate Eigenvector for Markov Ratings

In [18]:
if not np.all(P_markov == 0) and not np.any(np.isnan(P_markov)) and not np.any(np.isinf(P_markov)):

    eigvals_markov, eigvecs_markov = np.linalg.eig(P_markov.T)

    # Find the eigenvector corresponding to the eigenvalue closest to 1
    idx_markov = np.argmin(np.abs(eigvals_markov - 1.0))
    markov_ratings = eigvecs_markov[:, idx_markov].real

    # Standardize (non-negative and normalize to sum to 1)
    markov_ratings = np.abs(markov_ratings)
    markov_ratings = markov_ratings / markov_ratings.sum()

    markov_df = pd.DataFrame({
        "team": teams,
        "rating": markov_ratings
    }).sort_values("rating", ascending=False)

    print(markov_df.head(10))
else:
    print("Matrix P_markov is invalid (all zeros, contains NaN, or Inf values). Cannot compute Markov ratings.")

           team    rating
1     Argentina  0.037650
44      Uruguay  0.035985
5        Brazil  0.034839
4       Belgium  0.033370
27  Netherlands  0.032553
17      Germany  0.031200
31     Paraguay  0.030709
41  Switzerland  0.030666
40       Sweden  0.030610
16       France  0.030460


## 5. World Cup Simulation

Create a single function `simulate_world_cup` that can be used with any rating system (Massey, Keener, Markov). This function will handle:

1.  **Group Stage Advancement:** Identifying the top two teams from each group and the best eight third-placed teams.
2.  **Knockout Rounds:** Simulating matches based on the provided ratings until a single winner is determined.

In [19]:
import random

def simulate_world_cup(ratings_df, rating_column_name, groups, simulation_name):
    """
    Simulates the World Cup tournament based on given team ratings.

    Args:
        ratings_df (pd.DataFrame): DataFrame containing 'team' and the specified rating_column_name.
        rating_column_name (str): The name of the column in ratings_df that holds the ratings.
        groups (dict): A dictionary where keys are group names and values are lists of team names.
        simulation_name (str): A string identifier for the current simulation (e.g., 'Massey', 'Keener').

    Returns:
        str: The name of the winning team.
    """
    print(f"\n--- World Cup Simulation ({simulation_name} Ratings) ---")

    advancing_teams = {}
    third_placed_teams = []

    # --- Group Stage Advancement ---
    print(f"\n### Group Stage Advancement ({simulation_name} Ratings)")
    for group_name, team_list in groups.items():
        group_teams_ratings = ratings_df[ratings_df['team'].isin(team_list)].copy()
        group_teams_ratings = group_teams_ratings.sort_values(by=rating_column_name, ascending=False)

        # Top two teams from each group automatically advance
        advancing_teams[group_name] = group_teams_ratings.head(2)['team'].tolist()

        # Store third-placed teams for potential advancement
        if len(group_teams_ratings) >= 3:
            third_placed_teams.append({
                'team': group_teams_ratings.iloc[2]['team'],
                'rating': group_teams_ratings.iloc[2][rating_column_name],
                'group': group_name
            })

    # Sort third-placed teams by rating and take the top 8
    third_placed_teams_df = pd.DataFrame(third_placed_teams)
    if not third_placed_teams_df.empty:
        third_placed_teams_df = third_placed_teams_df.sort_values(by='rating', ascending=False).head(8)

    print(f"Teams advancing as 1st and 2nd in groups ({simulation_name}):")
    for group, teams_adv in advancing_teams.items():
       print(f"{group}: {teams_adv}")

    print(f"\nTeams advancing as best 3rd placed ({simulation_name}):")
    if not third_placed_teams_df.empty:
        print(third_placed_teams_df[['group', 'team', 'rating']].to_string(index=False))
    else:
        print("No third-placed teams to advance.")


    # --- Knockout Stage Simulation ---
    print(f"\n### Knockout Stage Simulation ({simulation_name} Ratings)")

    def simulate_match(team1_name, team2_name, ratings_df, rating_col):
        rating1 = ratings_df[ratings_df['team'] == team1_name][rating_col].iloc[0]
        rating2 = ratings_df[ratings_df['team'] == team2_name][rating_col].iloc[0]

        if rating1 > rating2:
            return team1_name
        elif rating2 > rating1:
            return team2_name
        else:
            # If ratings are equal, pick a random winner
            return random.choice([team1_name, team2_name])

    # Consolidate all advancing teams into a single list for Round of 32
    round_of_32_teams = []
    for group_teams in advancing_teams.values():
        round_of_32_teams.extend(group_teams)
    if not third_placed_teams_df.empty:
        round_of_32_teams.extend(third_placed_teams_df['team'].tolist())

    print(f"\nTotal teams in Round of 32 ({simulation_name}): {len(round_of_32_teams)}")
    print(f"Teams in Round of 32 ({simulation_name}):")
    print(round_of_32_teams)

    # Shuffle teams to create initial pairings (for a simplified bracket)
    random.shuffle(round_of_32_teams)

    current_round_teams = round_of_32_teams
    round_number = 32

    while len(current_round_teams) > 1:
        print(f"\n--- Round of {round_number} ({simulation_name}) ---")
        winners = []

        # Ensure even number of teams for pairings, if odd, one team gets a bye
        if len(current_round_teams) % 2 != 0:
            print(f"Warning: Odd number of teams ({len(current_round_teams)}). One team will get a bye.")
            winners.append(current_round_teams.pop(0)) # Give a bye to the first team

        for i in range(0, len(current_round_teams), 2):
            team1 = current_round_teams[i]
            team2 = current_round_teams[i+1]

            winner = simulate_match(team1, team2, ratings_df, rating_column_name)
            print(f"{team1} vs {team2} -> Winner: {winner}")
            winners.append(winner)

        current_round_teams = winners
        round_number //= 2

    print(f"\n--- World Cup Winner ({simulation_name}) ---")
    print(current_round_teams[0])
    return current_round_teams[0]

### Run Simulations with the General Function

In [20]:
# Run Massey Simulation
massey_winner = simulate_world_cup(massey_df, 'massey', groups, 'Massey')


--- World Cup Simulation (Massey Ratings) ---

### Group Stage Advancement (Massey Ratings)
Teams advancing as 1st and 2nd in groups (Massey):
Group A: ['Czech Republic', 'Mexico']
Group B: ['Switzerland', 'Canada']
Group C: ['Brazil', 'Scotland']
Group D: ['Turkey', 'Paraguay']
Group E: ['Germany', 'Ivory Coast']
Group F: ['Netherlands', 'Sweden']
Group G: ['Belgium', 'Egypt']
Group H: ['Spain', 'Uruguay']
Group I: ['France', 'Norway']
Group J: ['Argentina', 'Austria']
Group K: ['Portugal', 'Colombia']
Group L: ['England', 'Croatia']

Teams advancing as best 3rd placed (Massey):
  group        team    rating
Group C     Morocco  0.148029
Group A South Korea -0.013025
Group F       Japan -0.022397
Group J     Algeria -0.072106
Group D   Australia -0.097537
Group L       Ghana -0.151991
Group E     Ecuador -0.273042
Group G        Iran -0.276050

### Knockout Stage Simulation (Massey Ratings)

Total teams in Round of 32 (Massey): 32
Teams in Round of 32 (Massey):
['Czech Republic', 'Me

In [21]:
# Run Keener Simulation
keener_winner = simulate_world_cup(keener_df, 'rating', groups, 'Keener')


--- World Cup Simulation (Keener Ratings) ---

### Group Stage Advancement (Keener Ratings)
Teams advancing as 1st and 2nd in groups (Keener):
Group A: ['Mexico', 'South Korea']
Group B: ['Switzerland', 'Qatar']
Group C: ['Brazil', 'Scotland']
Group D: ['Paraguay', 'Australia']
Group E: ['Germany', 'Ecuador']
Group F: ['Netherlands', 'Sweden']
Group G: ['Belgium', 'Egypt']
Group H: ['Uruguay', 'Spain']
Group I: ['France', 'Norway']
Group J: ['Argentina', 'Austria']
Group K: ['Portugal', 'Colombia']
Group L: ['England', 'Croatia']

Teams advancing as best 3rd placed (Keener):
  group         team   rating
Group F        Japan 0.018538
Group D       Turkey 0.015484
Group C      Morocco 0.013656
Group E  Ivory Coast 0.012430
Group H Saudi Arabia 0.012383
Group L        Ghana 0.011801
Group J      Algeria 0.011571
Group I         Iraq 0.010324

### Knockout Stage Simulation (Keener Ratings)

Total teams in Round of 32 (Keener): 32
Teams in Round of 32 (Keener):
['Mexico', 'South Korea', '

In [22]:
# Run Markov Simulation
markov_winner = simulate_world_cup(markov_df, 'rating', groups, 'Markov')


--- World Cup Simulation (Markov Ratings) ---

### Group Stage Advancement (Markov Ratings)
Teams advancing as 1st and 2nd in groups (Markov):
Group A: ['Mexico', 'South Korea']
Group B: ['Switzerland', 'Qatar']
Group C: ['Brazil', 'Scotland']
Group D: ['Paraguay', 'Australia']
Group E: ['Germany', 'Ecuador']
Group F: ['Netherlands', 'Sweden']
Group G: ['Belgium', 'Egypt']
Group H: ['Uruguay', 'Saudi Arabia']
Group I: ['France', 'Norway']
Group J: ['Argentina', 'Austria']
Group K: ['Colombia', 'Portugal']
Group L: ['England', 'Ghana']

Teams advancing as best 3rd placed (Markov):
  group        team   rating
Group F       Japan 0.025337
Group I        Iraq 0.022873
Group H       Spain 0.019858
Group E Ivory Coast 0.018595
Group C     Morocco 0.017762
Group G New Zealand 0.017562
Group J      Jordan 0.017217
Group D      Turkey 0.015485

### Knockout Stage Simulation (Markov Ratings)

Total teams in Round of 32 (Markov): 32
Teams in Round of 32 (Markov):
['Mexico', 'South Korea', 'Swit

### Comparative Summary of Top Teams by Rating Method

In [23]:
comparative_top_teams = pd.DataFrame({
    'Massey Ratings': massey_df.head(3)['team'].tolist(),
    'Keener Ratings': keener_df.head(3)['team'].tolist(),
    'Markov Ratings': markov_df.head(3)['team'].tolist()
})

print(f"Massey Winner: {massey_winner}")
print(f"Keener Winner: {keener_winner}")
print(f"Markov Winner: {markov_winner}")

display(comparative_top_teams)

Massey Winner: Brazil
Keener Winner: Argentina
Markov Winner: Argentina


,Massey Ratings,Keener Ratings,Markov Ratings
0,Brazil,Argentina,Argentina
1,England,Brazil,Uruguay
2,Spain,Germany,Brazil
